# Random Forest Regressor — Food Delivery Time Prediction

This notebook adds a Random Forest regression model alongside the Decision Tree, KNN, and linear models. It follows the same 80/20 train/test split and reports MAE, MSE, RMSE, and R². The model is tuned using cross-validation on the training split, then evaluated once on the held-out test split.

The target is `Time_taken_min`. `Average_Speed_kmph` is excluded because it can reveal information about the completed delivery. Order identifiers and the raw date are also excluded.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

RANDOM_STATE = 42
DATA_PATH = Path(r'D:\innomatics\Machine Learning\food delivary time prediction\Food_Delivery_Time_Prediction.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('Food_Delivery_Time_Prediction.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place Food_Delivery_Time_Prediction.csv beside this notebook or update DATA_PATH.')

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
display(df.head())

In [ ]:
# Match the target and leakage exclusion used by the other model notebooks.
target = 'Time_taken_min'
excluded = [col for col in ['Average_Speed_kmph', 'Order_ID', 'Order_Date'] if col in df.columns]
if target not in df.columns:
    raise KeyError(f'Required target column {target!r} is missing.')

X = df.drop(columns=[target, *excluded]).copy()
y = pd.to_numeric(df[target], errors='coerce')
valid_rows = y.notna() & np.isfinite(y)
X, y = X.loc[valid_rows].copy(), y.loc[valid_rows].astype(float)
print('Excluded columns:', excluded)
print('Target:', target)
print('Feature columns:', X.columns.tolist())
print('Rows available:', len(X))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Preserve the meaningful ordering of these categories. Any other text columns
# are treated as nominal categories and one-hot encoded.
ordinal_categories = {
    'Restaurant_Load': ['Low', 'Medium', 'High'],
    'Delivery_Distance_Category': ['Short', 'Medium', 'Long'],
    'Traffic_Level': ['Low', 'Moderate', 'High', 'Severe'],
    'Delivery_Priority': ['Normal', 'Priority', 'VIP'],
}
ordinal_cols = [col for col in ordinal_categories if col in X_train.columns]
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
nominal_cols = [col for col in X_train.columns if col not in numeric_cols + ordinal_cols]
print('Numerical:', numeric_cols)
print('Ordinal:', ordinal_cols)
print('Nominal:', nominal_cols)

In [ ]:
transformers = []
if numeric_cols:
    transformers.append(('numeric', SimpleImputer(strategy='median'), numeric_cols))
if nominal_cols:
    nominal_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])
    transformers.append(('nominal', nominal_pipe, nominal_cols))
if ordinal_cols:
    ordinal_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            categories=[ordinal_categories[col] for col in ordinal_cols],
            handle_unknown='use_encoded_value', unknown_value=-1
        )),
    ])
    transformers.append(('ordinal', ordinal_pipe, ordinal_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])
rf_pipeline

## Baseline Random Forest

Random Forests do not require feature scaling. Imputation and category encoding are fitted inside the pipeline on each training fold, which keeps preprocessing separate from the test data.

In [ ]:
baseline_model = rf_pipeline.set_params(model__n_estimators=200, model__min_samples_leaf=1)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R²': r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, y_pred_baseline)
print('Baseline Random Forest results')
pd.Series(baseline_metrics).round(4)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_baseline, alpha=0.35)
limits = [min(y_test.min(), y_pred_baseline.min()), max(y_test.max(), y_pred_baseline.max())]
plt.plot(limits, limits, '--', color='red')
plt.xlabel('Actual delivery time (min)')
plt.ylabel('Predicted delivery time (min)')
plt.title('Baseline Random Forest: actual vs predicted')
plt.tight_layout()
plt.show()

residuals_baseline = y_test - y_pred_baseline
plt.figure(figsize=(7, 4))
plt.scatter(y_pred_baseline, residuals_baseline, alpha=0.35)
plt.axhline(0, linestyle='--', color='red')
plt.xlabel('Predicted delivery time (min)')
plt.ylabel('Residual (actual - predicted)')
plt.title('Baseline Random Forest residuals')
plt.tight_layout()
plt.show()

## Tune the model

Randomized search selects settings by cross-validated MAE on the training split. The held-out test set is reserved for the final comparison. Increase `n_iter` for a broader search if compute time allows.

In [ ]:
param_distributions = {
    'model__n_estimators': [150, 250, 400],
    'model__max_depth': [None, 12, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': [1.0, 'sqrt', 0.7],
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)
random_search.fit(X_train, y_train)
print('Best parameters:', random_search.best_params_)
print(f'Best 5-fold CV MAE: {-random_search.best_score_:.3f} min')

In [ ]:
best_rf = random_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
tuned_metrics = regression_metrics(y_test, y_pred_tuned)

comparison = pd.DataFrame(
    [baseline_metrics, tuned_metrics],
    index=['Baseline Random Forest', 'Tuned Random Forest']
)
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred_tuned, alpha=0.35)
limits = [min(y_test.min(), y_pred_tuned.min()), max(y_test.max(), y_pred_tuned.max())]
axes[0].plot(limits, limits, '--', color='red')
axes[0].set(xlabel='Actual delivery time (min)', ylabel='Predicted delivery time (min)', title='Tuned Random Forest: actual vs predicted')
axes[1].scatter(y_pred_tuned, y_test - y_pred_tuned, alpha=0.35)
axes[1].axhline(0, linestyle='--', color='red')
axes[1].set(xlabel='Predicted delivery time (min)', ylabel='Residual (actual - predicted)', title='Tuned Random Forest residuals')
plt.tight_layout()
plt.show()

## Feature importance

These impurity-based importances are useful as a quick summary, but they can favor features with many possible values. Use permutation importance on held-out data when you need a more reliable feature comparison.

In [ ]:
fitted_preprocessor = best_rf.named_steps['preprocessor']
feature_names = fitted_preprocessor.get_feature_names_out()
importances = best_rf.named_steps['model'].feature_importances_
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
display(feature_importance.head(20).to_frame('Importance'))
feature_importance.head(20).sort_values().plot(kind='barh', figsize=(9, 7), title='Top Random Forest feature importances')
plt.xlabel('Impurity-based importance')
plt.tight_layout()
plt.show()

## Interpretation

Compare the tuned model's test MAE and RMSE with the Decision Tree, KNN, Linear, Ridge, and Lasso test results. Because the notebooks use the same target and random seed, those scores are broadly comparable as long as they use the same dataset rows and feature exclusions. Lower MAE and RMSE are better; R² closer to 1 indicates more variance explained.